[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dreameraiquest/IncidentIQ/blob/main/Multi_Agent_DevOps_Incident_Analysis_Suite_v6_Hackathon_MVP.ipynb)

# IncidentIQ — Multi-Agent DevOps Incident Analysis Suite v6

**Hackathon MVP notebook aligned to the official problem statement and Vikrant's simplification plan.**

This notebook demonstrates an app-style backend where users upload operational logs and the system:

1. parses and normalizes mixed DevOps logs,
2. detects incident signals,
3. builds simple incident candidates,
4. runs a compact multi-agent reasoning flow,
5. generates remediation, Slack/JIRA preview payloads, and cookbook checklists,
6. exports traceable Markdown/JSON/CSV reports.


## v6 Retained Implementation Shape

The notebook explicitly keeps the simplified pipeline agreed for v6:

```text
raw logs
  -> Parser/Signal Agent
  -> signals
  -> grouped by candidate category
  -> grouped by service
  -> grouped by 5-minute time window
  -> incident candidate
```

The candidate grouping key is intentionally simple:

```python
group_key = (
    category,
    service,
    five_minute_window,
)
```

The compact agent set is:

1. Parser/Signal Agent
2. Incident Analysis Agent
3. Remediation Agent
4. Critic/Safety Agent
5. Action Builder Agent

## v6 Simplification Decisions

v6 deliberately optimizes for a working, judge-friendly hackathon demo rather than production completeness.

### Kept
- Clean input/output boundaries so teammates can work independently.
- Placeholder hooks for UI, API, RAG, LangGraph, and n8n owners.
- Evidence-grounded outputs with source file and line references.
- RAG-style runbook context with local fallback.
- Preview-first Slack/JIRA payloads.
- Fallback mode that works without paid services or API keys.

### Simplified
- Replaced complex clustering with a deterministic **Incident Candidate Builder**.
- Reduced public contracts to the contracts teams actually need.
- Collapsed standalone `RAGResult`, `SignalEvidence`, and `ExportManifest` into simpler nested structures.
- Kept one shared response shape so Gradio, FastAPI, LangGraph, n8n, and reporting can integrate safely.

## Boundary Chain

```text
Gradio / FastAPI
  -> analyze_uploaded_logs(files_or_paths, options)
  -> run_analysis(AnalysisRequest)
  -> parse + normalize logs
  -> extract_signals
  -> build_incident_candidates
  -> retrieve_context
  -> run_incident_graph
  -> build_action_payloads
  -> export_results
  -> AnalysisResponse
```


## 0. Install Dependencies

Keep dependencies light. The notebook must run in fallback mode without paid APIs. LangGraph can be added later inside `langgraph_team_impl` without changing the public contracts.


In [ ]:
!pip -q install pandas pydantic requests python-dateutil
# Optional later if the graph owner upgrades the placeholder:
# !pip -q install langgraph langchain langchain-openai


## 1. Configuration

Default mode is safe and demo-friendly:

- `ENABLE_LLM_MODE=false` unless an API key is present.
- `ENABLE_RAG_MODE=true` with embedded runbooks.
- `ENABLE_PREVIEW_MODE=true` so Slack/JIRA are generated as payloads, not sent.
- `ENABLE_REAL_INTEGRATIONS=false` until approval + credentials exist.

For optional LLM mode in Colab:

```python
import os
os.environ["OPENROUTER_API_KEY"] = "your-key"
os.environ["MODEL_NAME"] = "openai/gpt-4o-mini"
```


In [ ]:
import os, json, re, csv, zipfile, shutil, hashlib, textwrap, requests
from pathlib import Path
from datetime import datetime
from dateutil import parser as dtparser
from typing import Any, Dict, List, Optional, Tuple
from pydantic import BaseModel, Field

# -----------------------------------------------------------------------------
# Runtime feature flags
# -----------------------------------------------------------------------------
# These flags make the demo predictable. A judge should be able to run this
# notebook without Slack, JIRA, OpenAI, OpenRouter, Weaviate, or a database.
# -----------------------------------------------------------------------------
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

DEFAULT_OPTIONS = {
    "enable_rag": os.getenv("ENABLE_RAG_MODE", "true").lower() == "true",
    "enable_llm": os.getenv("ENABLE_LLM_MODE", "false").lower() == "true" and bool(OPENROUTER_API_KEY),
    "preview_only": os.getenv("ENABLE_PREVIEW_MODE", "true").lower() == "true",
    "real_integrations": os.getenv("ENABLE_REAL_INTEGRATIONS", "false").lower() == "true",
    "run_evals": False,
    "max_candidates": 12,
    "max_evidence_per_candidate": 12,
    "candidate_window_minutes": 5,
}

WORK_ROOT = Path("/content/incident_suite_v6") if Path("/content").exists() else Path("/mnt/data/incident_suite_v6")
WORK_ROOT.mkdir(parents=True, exist_ok=True)


def to_dict(obj: Any) -> Dict[str, Any]:
    """Pydantic v1/v2 compatibility helper used by exports and placeholders."""
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "dict"):
        return obj.dict()
    return dict(obj)


## 2. v6 Shared Contracts

v6 keeps only the contracts that are useful for parallel work.

### Public API contracts
- `AnalysisRequest`
- `AnalysisResponse`

### Core runtime contracts
- `LogEvent`
- `SignalMatch`
- `EvidenceCluster`
- `IncidentResult`
- `GraphState`

### External action contracts
- `TicketPayload`
- `SlackPayload`
- `CookbookPayload`
- `ActionPayload`

Everything else is nested data. This is simpler than v5 while preserving clean team boundaries.

### Not public contracts in v6
- `RAGResult` is folded into `GraphState.rag_context`.
- `SignalEvidence` is folded into `SignalMatch` and `EvidenceCluster.evidence`.
- `ExportManifest` is folded into `AnalysisResponse.exports`.

This keeps v6 simpler while preserving the useful boundaries.


In [ ]:
class AnalysisRequest(BaseModel):
    """Input boundary used by Gradio/FastAPI to start a run."""
    run_id: str
    uploaded_paths: List[str]
    options: Dict[str, Any] = Field(default_factory=dict)


class LogEvent(BaseModel):
    """Normalized log line. Runtime never reads hidden labels or eval answers."""
    source_file: str
    line_no: int
    timestamp: Optional[str] = None
    level: str = "UNKNOWN"
    service: str = "unknown"
    message: str
    trace_id: Optional[str] = None
    raw_line: str
    parse_status: str = "parsed"
    metadata: Dict[str, Any] = Field(default_factory=dict)


class SignalMatch(BaseModel):
    """Deterministic signal generated from a LogEvent.

    Evidence fields are embedded directly here to avoid a separate public
    SignalEvidence contract. This keeps the handoff easier for the team.
    """
    signal_type: str
    rule_id: str
    weight: float
    severity_hint: str = "UNKNOWN"
    source_file: str
    line_no: int
    timestamp: Optional[str] = None
    service: str = "unknown"
    level: str = "UNKNOWN"
    message: str
    trace_id: Optional[str] = None


class EvidenceCluster(BaseModel):
    """Incident candidate produced by the v6 Incident Candidate Builder.

    Despite the class name, this is intentionally not advanced ML clustering.
    v6 groups deterministic signals by:
        candidate category + service + small time window.

    This is stable, explainable, and enough for the hackathon demo.
    """
    cluster_id: str
    candidate_category: str
    signature: str
    affected_services: List[str]
    start_ts: Optional[str] = None
    end_ts: Optional[str] = None
    signal_count: int
    total_weight: float
    severity_hint: str = "UNKNOWN"
    evidence: List[Dict[str, Any]] = Field(default_factory=list)
    rule_ids: List[str] = Field(default_factory=list)
    summary: str = ""


class IncidentResult(BaseModel):
    """Main multi-agent output per incident candidate."""
    incident_id: str
    cluster_id: str
    category: str
    severity: str
    confidence: float
    title: str
    affected_services: List[str]
    symptom_vs_cause: str
    timeline: Dict[str, Any]
    root_cause: Dict[str, Any]
    remediation: Dict[str, Any]
    critic_findings: List[str] = Field(default_factory=list)
    evidence_grounded: bool = True
    rag_sources: List[str] = Field(default_factory=list)


class TicketPayload(BaseModel):
    """JIRA/GitHub issue preview payload. n8n can send this after approval."""
    system: str = "jira"
    create_issue: bool = True
    project_key: str = "OPS"
    title: str
    priority: str
    labels: List[str]
    description: str
    evidence: List[str]
    approval_status: str


class SlackPayload(BaseModel):
    """Slack/Teams-style notification preview payload."""
    channel: str = "#devops-incidents"
    send_message: bool = True
    title: str
    body: str
    approval_status: str


class CookbookPayload(BaseModel):
    """Reusable checklist generated from the incident analysis."""
    title: str
    steps: List[str]
    validation: List[str]
    safety_notes: List[str]


class ActionPayload(BaseModel):
    """External action boundary for n8n.

    This remains preview-only unless approval and real integrations are enabled.
    """
    run_id: str
    approval_status: str
    incident_id: str
    severity: str
    category: str
    ticket: TicketPayload
    slack: SlackPayload
    cookbook: CookbookPayload


class GraphState(BaseModel):
    """Internal orchestration state for future LangGraph implementation.

    Teams can add real LangGraph nodes later while preserving this state shape.
    """
    run_id: str
    input_paths: List[str] = Field(default_factory=list)
    raw_files: List[str] = Field(default_factory=list)
    ground_truth_files: List[str] = Field(default_factory=list)
    events: List[LogEvent] = Field(default_factory=list)
    signals: List[SignalMatch] = Field(default_factory=list)
    clusters: List[EvidenceCluster] = Field(default_factory=list)
    rag_context: Dict[str, List[Dict[str, Any]]] = Field(default_factory=dict)
    incidents: List[IncidentResult] = Field(default_factory=list)
    action_payloads: List[ActionPayload] = Field(default_factory=list)
    exports: Dict[str, str] = Field(default_factory=dict)
    errors: List[Dict[str, Any]] = Field(default_factory=list)


class AnalysisResponse(BaseModel):
    """Final response returned to Gradio/FastAPI and written to JSON export."""
    run_id: str
    status: str
    highest_severity: str = "UNKNOWN"
    summary: Dict[str, Any]
    clusters: List[EvidenceCluster]
    incidents: List[IncidentResult]
    action_payloads: List[ActionPayload]
    reports: Dict[str, str] = Field(default_factory=dict)
    eval: Dict[str, Any] = Field(default_factory=lambda: {"enabled": False, "summary": None})


## 3. Team Placeholder Hooks

Each stakeholder can fill in only their cell/function. Until then, the notebook keeps working through deterministic fallback implementations.

| Stakeholder area | Function to replace | What they own |
|---|---|---|
| Gradio UI | `gradio_entrypoint_placeholder` | Upload UI, progress, report visualization |
| FastAPI | `fastapi_entrypoint_placeholder` | `/health`, `/analyze`, saved uploads |
| RAG | `rag_retrieval_team_impl` | Runbook/SOP retrieval |
| LangGraph | `langgraph_team_impl` | Real StateGraph, nodes, routing, retries |
| n8n / integrations | `n8n_dispatch_team_impl` | Slack/JIRA routing after approval |

**Rule:** Return `None` until your implementation is complete. The fallback will continue to work for everyone else.


In [ ]:
def gradio_entrypoint_placeholder(files_or_paths, options=None):
    """UI owner cell.

    Replace only this function when wiring Gradio. It should collect uploaded
    files and call analyze_uploaded_logs(...). The rest of the pipeline remains
    unchanged.
    """
    return analyze_uploaded_logs(files_or_paths, options=options)


def fastapi_entrypoint_placeholder(uploaded_paths, options=None):
    """API owner cell.

    Replace only this function when wiring FastAPI. The endpoint should save
    files, then pass the saved paths into analyze_uploaded_logs(...).
    """
    return analyze_uploaded_logs(uploaded_paths, options=options)


def rag_retrieval_team_impl(analysis_context: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """RAG owner cell.

    Input:
        {"run_id": str, "clusters": [EvidenceCluster as dict]}

    Expected output:
        {"cluster_id": [runbook_chunk_dict, ...], ...}

    Return None until the RAG implementation is ready.
    """
    return None


def langgraph_team_impl(graph_input: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """LangGraph owner cell.

    Input:
        {"run_id": str, "clusters": [...], "rag_context": {...}, "options": {...}}

    Expected output:
        {"run_id": str, "incidents": [IncidentResult-compatible dict, ...]}

    Return None until the real StateGraph is ready.
    """
    return None


def n8n_dispatch_team_impl(action_payload: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """n8n/JIRA/Slack owner cell.

    Input:
        ActionPayload as dict

    Expected output:
        {"dispatch_mode": "n8n", "jira_status": "...", "slack_status": "..."}

    Return None until n8n webhook dispatch is ready.
    """
    return None


## 4. Public Entry Point — `analyze_uploaded_logs`

This is the function Gradio or FastAPI should call. It accepts file paths, Colab upload objects, Gradio temp files, or file-like objects.


In [ ]:
def new_run_id() -> str:
    return "run_" + datetime.utcnow().strftime("%Y%m%d_%H%M%S")


def analyze_uploaded_logs(files_or_paths: List[Any], options: Optional[Dict[str, Any]] = None) -> AnalysisResponse:
    """Public upload boundary for Gradio/FastAPI/Colab.

    This function only handles input normalization and delegates all actual
    incident analysis to run_analysis(...). Keeping it small makes UI/API work
    independent from the pipeline internals.
    """
    opts = {**DEFAULT_OPTIONS, **(options or {})}
    run_id = new_run_id()

    uploaded_paths: List[str] = []
    run_upload_dir = WORK_ROOT / "uploads" / run_id
    run_upload_dir.mkdir(parents=True, exist_ok=True)

    for item in files_or_paths:
        if isinstance(item, (str, Path)):
            src = Path(item)
            dst = run_upload_dir / src.name
            if src.resolve() != dst.resolve():
                shutil.copy(src, dst)
            uploaded_paths.append(str(dst))
        elif hasattr(item, "name") and Path(str(item.name)).exists():
            src = Path(str(item.name))
            dst = run_upload_dir / src.name
            shutil.copy(src, dst)
            uploaded_paths.append(str(dst))
        elif hasattr(item, "read"):
            name = getattr(item, "name", f"upload_{len(uploaded_paths)}.log")
            dst = run_upload_dir / Path(str(name)).name
            data = item.read()
            if isinstance(data, str):
                data = data.encode("utf-8")
            dst.write_bytes(data)
            uploaded_paths.append(str(dst))
        else:
            raise TypeError(f"Unsupported upload object: {type(item)}")

    request = AnalysisRequest(run_id=run_id, uploaded_paths=uploaded_paths, options=opts)
    return run_analysis(request)


## 5. Main Pipeline Boundary — `run_analysis`

This is the stable backend contract. It mirrors Vikrant's simplified shared response object while keeping enough structure for team development.


In [ ]:
def run_analysis(request: AnalysisRequest) -> AnalysisResponse:
    """Run the full hackathon MVP pipeline.

    Pipeline stages:
    1. Prepare inputs and safely extract ZIPs.
    2. Parse raw logs into LogEvent objects.
    3. Extract deterministic SignalMatch records.
    4. Build simple incident candidates.
    5. Retrieve RAG/runbook context.
    6. Run compact multi-agent reasoning.
    7. Build preview-first Slack/JIRA/cookbook payloads.
    8. Export Markdown, JSON, and CSV reports.

    Important boundary:
    - Runtime parsing skips ground_truth_eval_only.
    - Eval data is detected but not used by agents.
    """
    run_root = WORK_ROOT / "runs" / request.run_id
    input_root = run_root / "input"
    output_root = run_root / "outputs"
    input_root.mkdir(parents=True, exist_ok=True)
    output_root.mkdir(parents=True, exist_ok=True)

    state = GraphState(run_id=request.run_id, input_paths=request.uploaded_paths)

    raw_files, ground_truth_files = prepare_inputs(request.uploaded_paths, input_root)
    state.raw_files = [str(p) for p in raw_files]
    state.ground_truth_files = [str(p) for p in ground_truth_files]

    for path in raw_files:
        try:
            state.events.extend(parse_log_file(path, input_root))
        except Exception as exc:
            state.errors.append({"stage": "parse", "file": str(path), "error": str(exc)})

    state.signals = extract_signals(state.events)
    state.clusters = build_incident_candidates(
        state.signals,
        window_minutes=request.options.get("candidate_window_minutes", 5),
        max_evidence_per_candidate=request.options.get("max_evidence_per_candidate", 12),
        max_candidates=request.options.get("max_candidates", 12),
    )

    analysis_context = {
        "run_id": request.run_id,
        "clusters": [to_dict(c) for c in state.clusters],
    }
    state.rag_context = retrieve_context(analysis_context) if request.options.get("enable_rag", True) else {}

    graph_input = {
        "run_id": request.run_id,
        "clusters": [to_dict(c) for c in state.clusters],
        "rag_context": state.rag_context,
        "options": request.options,
    }
    graph_output = run_incident_graph(graph_input)
    state.incidents = [IncidentResult(**item) for item in graph_output.get("incidents", [])]

    approval_status = "PREVIEW_ONLY" if request.options.get("preview_only", True) else "PENDING_HUMAN_APPROVAL"
    state.action_payloads = [build_action_payload(request.run_id, incident, approval_status) for incident in state.incidents]

    state.exports = export_results(output_root, state)

    highest = highest_severity([incident.severity for incident in state.incidents])
    return AnalysisResponse(
        run_id=request.run_id,
        status="completed" if not state.errors else "completed_with_warnings",
        highest_severity=highest,
        summary={
            "raw_files": len(state.raw_files),
            "ground_truth_files_detected_but_not_used": len(state.ground_truth_files),
            "events_parsed": len(state.events),
            "signals_found": len(state.signals),
            "incident_candidates_found": len(state.clusters),
            "incidents_found": len(state.incidents),
            "errors": state.errors,
        },
        clusters=state.clusters,
        incidents=state.incidents,
        action_payloads=state.action_payloads,
        reports=state.exports,
        eval={"enabled": False, "summary": None},
    )


def highest_severity(severities: List[str]) -> str:
    order = {"P1": 1, "P2": 2, "P3": 3, "P4": 4, "UNKNOWN": 99}
    if not severities:
        return "UNKNOWN"
    return sorted(severities, key=lambda s: order.get(str(s).upper(), 99))[0]


## 6. Ingestion and Parsing

Owner target later: `src/ingest/`

This section is deterministic. It accepts `.zip`, `.jsonl`, `.log`, and `.txt`, safely extracts ZIPs, skips hidden eval files during runtime, and preserves source file + line number for evidence traceability.


In [ ]:
ALLOWED_EXTENSIONS = {".jsonl", ".log", ".txt"}


def safe_extract_zip(zip_path: Path, dest: Path) -> None:
    """Extract ZIP safely and prevent path traversal."""
    with zipfile.ZipFile(zip_path, "r") as zf:
        for member in zf.infolist():
            target = (dest / member.filename).resolve()
            if not str(target).startswith(str(dest.resolve())):
                raise ValueError(f"Unsafe zip path: {member.filename}")
        zf.extractall(dest)


def prepare_inputs(uploaded_paths: List[str], input_root: Path) -> Tuple[List[Path], List[Path]]:
    """Copy/extract uploads and return runtime raw files plus eval-only files."""
    input_root.mkdir(parents=True, exist_ok=True)

    for raw_path in uploaded_paths:
        src = Path(raw_path)
        if not src.exists():
            raise FileNotFoundError(src)
        dst = input_root / src.name
        if src.resolve() != dst.resolve():
            shutil.copy(src, dst)
        if dst.suffix.lower() == ".zip":
            safe_extract_zip(dst, input_root / dst.stem)

    raw_files: List[Path] = []
    ground_truth_files: List[Path] = []

    for path in input_root.rglob("*"):
        if not path.is_file():
            continue
        normalized = str(path).replace("\\", "/").lower()
        if "ground_truth_eval_only" in normalized or "golden" in path.name.lower() or "eval_cases" in path.name.lower():
            ground_truth_files.append(path)
            continue
        if path.suffix.lower() in ALLOWED_EXTENSIONS:
            raw_files.append(path)

    return sorted(raw_files), sorted(ground_truth_files)


def parse_timestamp(value: Any) -> Optional[str]:
    if value in (None, ""):
        return None
    try:
        return dtparser.parse(str(value)).isoformat()
    except Exception:
        return None


def infer_level(message: str, data: Optional[Dict[str, Any]] = None) -> str:
    if data:
        for key in ["level", "severity", "log_level"]:
            if data.get(key):
                return str(data[key]).upper()
    msg = message.lower()
    if any(x in msg for x in ["error", "exception", "failed", "timeout", "denied", "oom", "critical"]):
        return "ERROR"
    if any(x in msg for x in ["warn", "warning", "retry", "degraded"]):
        return "WARN"
    return "INFO"


def infer_service(message: str, data: Optional[Dict[str, Any]] = None) -> str:
    if data:
        for key in ["service", "app", "component", "logger", "pod", "container", "namespace"]:
            if data.get(key):
                return str(data[key])
    match = re.search(r"\b([a-z][a-z0-9-]{2,40}-(api|svc|worker|consumer|gateway|db|cache))\b", message.lower())
    return match.group(1) if match else "unknown"


def parse_log_file(path: Path, input_root: Path) -> List[LogEvent]:
    """Parse JSONL or plain text logs into normalized LogEvent records."""
    rel = str(path.relative_to(input_root)) if path.is_relative_to(input_root) else path.name
    events: List[LogEvent] = []

    with open(path, "r", encoding="utf-8", errors="replace") as handle:
        for line_no, raw in enumerate(handle, start=1):
            raw = raw.rstrip("\n")
            if not raw.strip():
                continue

            data = None
            message = raw
            parse_status = "parsed_text"

            if raw.lstrip().startswith("{"):
                try:
                    data = json.loads(raw)
                    parse_status = "parsed_json"
                    message = str(data.get("message") or data.get("msg") or data.get("log") or raw)
                except Exception:
                    parse_status = "malformed_json_kept_as_text"

            timestamp = None
            trace_id = None
            if data:
                timestamp = parse_timestamp(data.get("timestamp") or data.get("ts") or data.get("time"))
                trace_id = data.get("trace_id") or data.get("traceId") or data.get("correlation_id")

            events.append(LogEvent(
                source_file=rel,
                line_no=line_no,
                timestamp=timestamp,
                level=infer_level(message, data),
                service=infer_service(message, data),
                message=message[:2000],
                trace_id=str(trace_id) if trace_id else None,
                raw_line=raw[:4000],
                parse_status=parse_status,
                metadata={"json_keys": sorted(list(data.keys())) if isinstance(data, dict) else []},
            ))

    return events


## 7. Parser/Signal Agent and Incident Candidate Builder

Owner target later: `src/signals/` and `src/clustering/`.

v6 intentionally avoids complex clustering. The deterministic Parser/Signal Agent produces signals, then the Incident Candidate Builder groups them into incident candidates using exactly this boundary:

```text
raw logs
  -> parsed LogEvent records
  -> SignalMatch records
  -> grouped by candidate category
  -> grouped by service
  -> grouped by 5-minute time window
  -> EvidenceCluster incident candidate
```

Implementation rule:

```python
group_key = (
    category,
    service,
    five_minute_window,
)
```

This is easy to explain in the demo, stable enough for the hackathon, and keeps the notebook aligned with the problem statement: upload operational logs, classify/detect issues, recommend remediation, and generate Slack/JIRA/cookbook outputs.


In [ ]:
SIGNAL_RULES = [
    ("Database", "db_hikari_timeout", 4.0, "P1", r"hikaripool|connection pool|too many clients|lock wait|deadlock|slow query|pg_stat_activity"),
    ("Network", "network_connectivity", 3.0, "P2", r"dns failure|connection reset|connection refused|tls handshake|tcp retransmit|network unreachable"),
    ("Authentication", "auth_failure", 3.0, "P2", r"jwt|invalid signature|issuer mismatch|jwk|401|403|rbac|unauthorized|forbidden"),
    ("Memory/CPU", "resource_pressure", 4.0, "P1", r"oomkilled|out of memory|heap|gc pause|cpu throttl|restart loop|memory pressure"),
    ("Deployment regression", "deployment_regression", 4.0, "P1", r"rollout|deployment|canary|rollback|feature flag|configmap|new version|release"),
    ("API timeout", "api_timeout", 3.0, "P2", r"504|gateway timeout|upstream timed out|p95|latency breach|circuit breaker|request timeout"),
    ("Queue backlog", "queue_backlog", 3.0, "P2", r"consumer lag|dlq|dead letter|retry topic|poison message|rebalance|queue backlog"),
    ("Disk/storage", "disk_storage", 4.0, "P1", r"no space left|disk pressure|logrotate failed|io wait|persistent volume|volume full|storage latency"),
    ("External dependency", "external_dependency", 3.0, "P2", r"vendor|provider|third-party|429|retry-after|external dependency|feed delayed"),
    ("Unknown", "unknown_incomplete_evidence", 1.0, "P3", r"missing correlation|incomplete trace|redacted|unexpected state|unknown error"),
]

COMPILED_RULES = [(cat, rule, weight, sev, re.compile(pattern, re.I)) for cat, rule, weight, sev, pattern in SIGNAL_RULES]


def extract_signals(events: List[LogEvent]) -> List[SignalMatch]:
    """Run deterministic regex rules over normalized log events."""
    matches: List[SignalMatch] = []
    for event in events:
        haystack = f"{event.level} {event.service} {event.message} {event.raw_line}"
        for category, rule_id, weight, severity_hint, regex in COMPILED_RULES:
            if regex.search(haystack):
                matches.append(SignalMatch(
                    signal_type=category,
                    rule_id=rule_id,
                    weight=weight,
                    severity_hint=severity_hint,
                    source_file=event.source_file,
                    line_no=event.line_no,
                    timestamp=event.timestamp,
                    service=event.service,
                    level=event.level,
                    message=event.message,
                    trace_id=event.trace_id,
                ))
    return matches


def bucket_timestamp(ts: Optional[str], window_minutes: int = 5) -> str:
    """Return a simple time bucket for candidate grouping."""
    if not ts:
        return "no_timestamp"
    try:
        dt = dtparser.parse(ts)
        minute_bucket = (dt.minute // window_minutes) * window_minutes
        return dt.replace(minute=minute_bucket, second=0, microsecond=0).isoformat()
    except Exception:
        return "bad_timestamp"


def severity_from_group(matches: List[SignalMatch]) -> str:
    order = {"P1": 1, "P2": 2, "P3": 3, "P4": 4, "UNKNOWN": 99}
    return sorted([m.severity_hint for m in matches], key=lambda s: order.get(str(s).upper(), 99))[0]


def build_incident_candidates(
    signals: List[SignalMatch],
    window_minutes: int = 5,
    max_evidence_per_candidate: int = 12,
    max_candidates: int = 12,
) -> List[EvidenceCluster]:
    """Build explainable incident candidates from deterministic signals.

    v6 simplification:
    - no embedding clustering
    - no dependency graph inference
    - no trace graph construction
    - no causal ML

    Instead we group by category, service, and a small time bucket. This gives
    the reasoning agents enough evidence while keeping the demo reliable.
    """
    grouped: Dict[Tuple[str, str, str], List[SignalMatch]] = {}
    for signal in signals:
        bucket = bucket_timestamp(signal.timestamp, window_minutes=window_minutes)
        # v6 contract: candidate_category + service + 5-minute window.
        # Keep this explicit so every stakeholder can see the grouping boundary.
        group_key = (signal.signal_type, signal.service or "unknown", bucket)
        grouped.setdefault(group_key, []).append(signal)

    candidates: List[EvidenceCluster] = []
    for (category, service, bucket), group in grouped.items():
        group_sorted = sorted(group, key=lambda s: (s.timestamp or "", s.source_file, s.line_no))
        total_weight = sum(s.weight for s in group_sorted)
        evidence = [to_dict(s) for s in group_sorted[:max_evidence_per_candidate]]
        first = group_sorted[0]
        signature = first.rule_id
        cluster_hash = hashlib.md5(f"{category}|{service}|{bucket}|{signature}".encode()).hexdigest()[:8]
        cluster_id = f"candidate_{cluster_hash}"

        candidates.append(EvidenceCluster(
            cluster_id=cluster_id,
            candidate_category=category,
            signature=signature,
            affected_services=sorted({s.service for s in group_sorted if s.service}),
            start_ts=group_sorted[0].timestamp,
            end_ts=group_sorted[-1].timestamp,
            signal_count=len(group_sorted),
            total_weight=round(total_weight, 2),
            severity_hint=severity_from_group(group_sorted),
            evidence=evidence,
            rule_ids=sorted({s.rule_id for s in group_sorted}),
            summary=f"{category} signals for {service} in bucket {bucket}; {len(group_sorted)} matching lines.",
        ))

    # Prioritize high signal weight and evidence volume for demo clarity.
    candidates.sort(key=lambda c: (c.total_weight, c.signal_count), reverse=True)
    return candidates[:max_candidates]


## 8. RAG Context Retrieval

Owner target later: `src/rag/`.

v6 treats RAG as context attached to `GraphState.rag_context`, not as a separate public contract. This makes the interface simpler while still showing RAG in the demo.


In [ ]:
RUNBOOKS = [
    {
        "title": "Database connection pool exhaustion",
        "incident_type": "Database",
        "symptoms": ["HikariPool", "too many clients", "lock wait", "slow query"],
        "diagnostics": ["Check DB active sessions", "Inspect lock waits", "Review DB connection pool metrics"],
        "remediation": ["Reduce request pressure", "Tune connection pool only after DB capacity check", "Identify long-running transactions"],
        "validation": ["DB wait time returns to baseline", "5xx rate drops", "Pool acquisition latency normalizes"],
        "safety_notes": ["Do not restart DB primary blindly", "Avoid increasing pool size without DB capacity review"],
        "owner": "sre",
        "source": "embedded_runbook",
    },
    {
        "title": "Network or DNS instability",
        "incident_type": "Network",
        "symptoms": ["DNS failure", "connection reset", "TLS handshake timeout"],
        "diagnostics": ["Check upstream health", "Validate DNS resolution", "Review network error rate"],
        "remediation": ["Route traffic to healthy upstream", "Rollback bad network config", "Escalate provider issue if external"],
        "validation": ["Connection errors drop", "DNS success rate normalizes", "Latency returns to baseline"],
        "safety_notes": ["Avoid broad firewall changes without approval"],
        "owner": "platform",
        "source": "embedded_runbook",
    },
    {
        "title": "Authentication or token validation incident",
        "incident_type": "Authentication",
        "symptoms": ["JWT", "invalid signature", "issuer mismatch", "401", "403"],
        "diagnostics": ["Check auth config", "Validate issuer and audience", "Inspect JWK refresh status"],
        "remediation": ["Restore correct auth configuration", "Refresh keys", "Rollback bad RBAC policy"],
        "validation": ["401/403 spike drops", "Token validation succeeds", "Login success rate recovers"],
        "safety_notes": ["Do not disable auth checks globally"],
        "owner": "identity",
        "source": "embedded_runbook",
    },
    {
        "title": "Resource pressure: memory or CPU",
        "incident_type": "Memory/CPU",
        "symptoms": ["OOMKilled", "GC pause", "CPU throttling", "restart loop"],
        "diagnostics": ["Check pod restarts", "Review memory and CPU metrics", "Inspect recent traffic changes"],
        "remediation": ["Scale service if safe", "Rollback memory-heavy release", "Tune resource limits"],
        "validation": ["Restart count stabilizes", "CPU throttling decreases", "Latency returns to baseline"],
        "safety_notes": ["Avoid repeated restarts without identifying memory pressure source"],
        "owner": "sre",
        "source": "embedded_runbook",
    },
    {
        "title": "Deployment regression",
        "incident_type": "Deployment regression",
        "symptoms": ["rollout", "canary", "new version", "feature flag", "rollback"],
        "diagnostics": ["Check deployment timeline", "Compare old vs new version errors", "Review feature flag changes"],
        "remediation": ["Rollback candidate release", "Disable risky feature flag", "Pause rollout"],
        "validation": ["Error rate returns to pre-deploy baseline", "Canary health passes"],
        "safety_notes": ["Validate rollback impact before applying"],
        "owner": "release",
        "source": "embedded_runbook",
    },
    {
        "title": "API timeout or latency cascade",
        "incident_type": "API timeout",
        "symptoms": ["504", "gateway timeout", "upstream timed out", "circuit breaker"],
        "diagnostics": ["Check downstream latency", "Review circuit breaker state", "Inspect p95/p99 latency"],
        "remediation": ["Reduce timeout cascade", "Scale bottleneck service", "Rollback recent slow dependency change"],
        "validation": ["504 rate drops", "p95 latency normalizes", "Circuit breaker closes"],
        "safety_notes": ["Do not simply raise timeouts without identifying downstream bottleneck"],
        "owner": "api-platform",
        "source": "embedded_runbook",
    },
]


def score_runbook(cluster: Dict[str, Any], runbook: Dict[str, Any]) -> float:
    """Lightweight lexical retrieval score for hackathon fallback RAG."""
    text = " ".join([
        cluster.get("candidate_category", ""),
        cluster.get("signature", ""),
        " ".join(cluster.get("affected_services", [])),
        " ".join(e.get("message", "") for e in cluster.get("evidence", [])),
    ]).lower()

    score = 0.0
    if runbook["incident_type"].lower() == cluster.get("candidate_category", "").lower():
        score += 3.0
    for symptom in runbook.get("symptoms", []):
        if symptom.lower() in text:
            score += 1.0
    return score


def retrieve_context(analysis_context: Dict[str, Any]) -> Dict[str, List[Dict[str, Any]]]:
    """RAG boundary.

    Returns a dict keyed by cluster_id:
        {cluster_id: [runbook_chunk_dict, ...]}
    """
    team_result = rag_retrieval_team_impl(analysis_context)
    if team_result is not None:
        return team_result

    rag_context: Dict[str, List[Dict[str, Any]]] = {}
    for cluster in analysis_context.get("clusters", []):
        scored = []
        for runbook in RUNBOOKS:
            rb = dict(runbook)
            rb["score"] = score_runbook(cluster, runbook)
            if rb["score"] > 0:
                scored.append(rb)
        scored.sort(key=lambda x: x["score"], reverse=True)
        rag_context[cluster.get("cluster_id")] = scored[:3]
    return rag_context


## 9. Compact Multi-Agent Reasoning Boundary

Owner target later: `src/graph/` and `src/agents/`.

v6 keeps **4-5 visible agents**, not a large production-style agent mesh:

```text
Parser/Signal Agent
  -> Incident Analysis Agent
  -> Remediation Agent with RAG context
  -> Critic/Safety Agent
  -> Action Builder Agent
```

Responsibilities:

1. **Parser/Signal Agent** — deterministic Python for parsing, normalization, and signal matching.
2. **Incident Analysis Agent** — classification, severity, symptom-vs-cause, timeline, and RCA.
3. **Remediation Agent** — RAG-backed remediation, validation checks, and safety notes.
4. **Critic/Safety Agent** — evidence grounding, hallucination risk, and unsafe action checks.
5. **Action Builder Agent** — Slack preview, JIRA preview, and cookbook checklist.

This still demonstrates multi-agent collaboration while reducing implementation risk.


In [ ]:
def openrouter_chat_json(system_prompt: str, user_payload: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """Optional LLM helper. Fallback mode does not require this."""
    if not OPENROUTER_API_KEY:
        return None
    try:
        response = requests.post(
            f"{OPENROUTER_BASE_URL}/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": MODEL_NAME,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
                ],
                "temperature": 0.1,
            },
            timeout=45,
        )
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"]
        match = re.search(r"\{.*\}", content, re.S)
        return json.loads(match.group(0) if match else content)
    except Exception as exc:
        print(f"LLM call failed; using fallback. Error: {exc}")
        return None


def evidence_refs(cluster: Dict[str, Any], limit: int = 8) -> List[str]:
    refs = []
    for ev in cluster.get("evidence", [])[:limit]:
        refs.append(f"{ev.get('source_file')}:{ev.get('line_no')} - {ev.get('message')}")
    return refs


def fallback_incident_analysis_agent(cluster: Dict[str, Any], rag_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Incident Analysis Agent fallback.

    Owns classification, severity, timeline, symptom-vs-cause, and RCA draft.
    """
    category = cluster.get("candidate_category", "Unknown")
    severity = cluster.get("severity_hint", "UNKNOWN")
    services = cluster.get("affected_services", [])
    refs = evidence_refs(cluster)

    root_cause = {
        "primary_root_cause": f"Likely {category} incident indicated by {cluster.get('signature')} signals.",
        "alternative_causes": ["Recent deployment/config change", "External dependency degradation", "Insufficient telemetry"],
        "supporting_evidence": refs,
        "missing_evidence": ["Metrics", "Distributed traces", "Recent deployment history"],
        "confidence": 0.72 if refs else 0.4,
    }
    timeline = {
        "events": cluster.get("evidence", [])[:8],
        "interpretation": f"{category} signals were observed for {', '.join(services) or 'unknown service'}."
    }
    return {
        "category": category,
        "severity": severity,
        "confidence": root_cause["confidence"],
        "title": f"{severity} {category} incident candidate",
        "symptom_vs_cause": f"{category} is the strongest deterministic candidate. Related timeout/error lines may be downstream symptoms, not necessarily the root cause.",
        "timeline": timeline,
        "root_cause": root_cause,
        "rag_sources": [chunk.get("title", "runbook") for chunk in rag_chunks],
    }


def fallback_remediation_agent(analysis: Dict[str, Any], rag_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Remediation Agent fallback.

    Uses retrieved runbook chunks when available. Otherwise returns safe generic
    diagnostics and validation steps.
    """
    best = rag_chunks[0] if rag_chunks else {}
    return {
        "immediate_actions": best.get("diagnostics") or ["Review supporting evidence", "Check service health", "Validate recent changes"],
        "validation_checks": best.get("validation") or ["Error rate decreases", "Latency returns to baseline", "No new critical alerts appear"],
        "safety_notes": best.get("safety_notes") or ["Human approval recommended before disruptive actions"],
        "requires_human_approval": True,
    }


def fallback_critic_agent(analysis: Dict[str, Any], remediation: Dict[str, Any]) -> List[str]:
    """Critic/Safety Agent fallback.

    Checks that the output remains evidence-grounded and preview-safe.
    """
    findings = []
    if not analysis.get("root_cause", {}).get("supporting_evidence"):
        findings.append("Weak evidence: no source line references were available.")
    else:
        findings.append("Evidence grounding check passed: RCA includes source line references.")
    if remediation.get("requires_human_approval"):
        findings.append("Safety check passed: remediation requires human approval before disruptive action.")
    findings.append("Preview mode recommended for Slack/JIRA until approval and credentials are present.")
    return findings


def fallback_incident(cluster: Dict[str, Any], rag_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    analysis = fallback_incident_analysis_agent(cluster, rag_chunks)
    remediation = fallback_remediation_agent(analysis, rag_chunks)
    critic_findings = fallback_critic_agent(analysis, remediation)
    return {
        "incident_id": cluster.get("cluster_id"),
        "cluster_id": cluster.get("cluster_id"),
        "category": analysis["category"],
        "severity": analysis["severity"],
        "confidence": analysis["confidence"],
        "title": analysis["title"],
        "affected_services": cluster.get("affected_services", []),
        "symptom_vs_cause": analysis["symptom_vs_cause"],
        "timeline": analysis["timeline"],
        "root_cause": analysis["root_cause"],
        "remediation": remediation,
        "critic_findings": critic_findings,
        "evidence_grounded": bool(analysis.get("root_cause", {}).get("supporting_evidence")),
        "rag_sources": analysis.get("rag_sources", []),
    }


def run_incident_graph(graph_input: Dict[str, Any]) -> Dict[str, Any]:
    """LangGraph boundary.

    The graph owner can replace this through langgraph_team_impl(...). Until
    then, this function runs a compact sequential multi-agent fallback.
    """
    team_result = langgraph_team_impl(graph_input)
    if team_result is not None:
        return team_result

    incidents = []
    use_llm = graph_input.get("options", {}).get("enable_llm", False)
    rag_context = graph_input.get("rag_context", {})

    for cluster in graph_input.get("clusters", []):
        rag_chunks = rag_context.get(cluster.get("cluster_id"), [])
        base = fallback_incident(cluster, rag_chunks)

        if use_llm and OPENROUTER_API_KEY:
            system_prompt = """
You are an SRE incident analysis agent. Return ONLY valid JSON with these keys:
category, severity, confidence, title, symptom_vs_cause, root_cause, remediation, critic_findings.
Rules:
- Use only provided evidence and RAG context.
- Cite evidence as file:line - message.
- Do not read or infer hidden ground truth.
- If evidence is weak, say so.
- Recommend safe actions only.
"""
            llm_result = openrouter_chat_json(system_prompt, {"cluster": cluster, "rag_chunks": rag_chunks})
            if llm_result:
                for key in ["category", "severity", "confidence", "title", "symptom_vs_cause", "root_cause", "remediation", "critic_findings"]:
                    if key in llm_result:
                        base[key] = llm_result[key]

        incidents.append(base)

    return {"run_id": graph_input.get("run_id"), "incidents": incidents}


## 10. Action Payload Builder and n8n Dispatch Boundary

Owner target later: `src/reporting/`, `src/adapters/`, or n8n workflow JSON.

This section generates Slack/JIRA/cookbook previews. Real sending remains disabled unless explicitly implemented and approved.


In [ ]:
def build_action_payload(run_id: str, incident: IncidentResult, approval_status: str = "PREVIEW_ONLY") -> ActionPayload:
    """Build preview-ready action payloads from one incident result."""
    evidence = incident.root_cause.get("supporting_evidence", [])
    actions = incident.remediation.get("immediate_actions", [])
    validation = incident.remediation.get("validation_checks", [])
    safety = incident.remediation.get("safety_notes", [])

    ticket = TicketPayload(
        title=f"{incident.severity}: {incident.category} - {incident.title}",
        priority=incident.severity,
        labels=["ai-generated", "incident", incident.severity.lower(), incident.category.lower().replace("/", "-").replace(" ", "-")],
        description=(
            f"Root cause: {incident.root_cause.get('primary_root_cause')}\n\n"
            f"Evidence:\n- " + "\n- ".join(evidence[:10]) + "\n\n"
            f"Recommended actions:\n- " + "\n- ".join(actions[:10]) + "\n\n"
            f"Safety notes:\n- " + "\n- ".join(safety[:10])
        ),
        evidence=evidence,
        approval_status=approval_status,
    )

    slack = SlackPayload(
        title=f"{incident.severity}: {incident.category} incident candidate",
        body=(
            f"Affected services: {', '.join(incident.affected_services) or 'unknown'}\n"
            f"Likely cause: {incident.root_cause.get('primary_root_cause')}\n"
            f"Top action: {(actions or ['Review evidence'])[0]}\n"
            f"Status: {approval_status}"
        ),
        approval_status=approval_status,
    )

    cookbook = CookbookPayload(
        title=f"Cookbook: {incident.category} / {incident.cluster_id}",
        steps=actions,
        validation=validation,
        safety_notes=safety,
    )

    return ActionPayload(
        run_id=run_id,
        approval_status=approval_status,
        incident_id=incident.incident_id,
        severity=incident.severity,
        category=incident.category,
        ticket=ticket,
        slack=slack,
        cookbook=cookbook,
    )


def dispatch_actions(action_payload: ActionPayload) -> Dict[str, Any]:
    """n8n dispatch boundary.

    Current behavior is preview-only. n8n owner can implement the placeholder
    without changing ActionPayload or IncidentResult.
    """
    payload_dict = to_dict(action_payload)
    team_result = n8n_dispatch_team_impl(payload_dict)
    if team_result is not None:
        return team_result

    return {
        "run_id": payload_dict.get("run_id"),
        "incident_id": payload_dict.get("incident_id"),
        "dispatch_mode": "preview_only",
        "jira_status": "not_sent",
        "slack_status": "not_sent",
        "message": "Payload is ready for n8n/JIRA/Slack preview. No external systems were called.",
    }


## 11. Reporting and Exports

Owner target later: `src/reporting/`.

Reports prove the system is traceable and actionable: Markdown for humans, JSON for integration, CSV for quick review.


In [ ]:
def export_results(output_root: Path, state: GraphState) -> Dict[str, str]:
    output_root.mkdir(parents=True, exist_ok=True)

    json_path = output_root / "analysis_response.json"
    md_path = output_root / "incident_report.md"
    csv_path = output_root / "incidents.csv"

    payload = {
        "run_id": state.run_id,
        "summary": {
            "events_parsed": len(state.events),
            "signals_found": len(state.signals),
            "incident_candidates_found": len(state.clusters),
            "incidents_found": len(state.incidents),
            "ground_truth_files_detected_but_not_used": len(state.ground_truth_files),
            "errors": state.errors,
        },
        "clusters": [to_dict(c) for c in state.clusters],
        "rag_context": state.rag_context,
        "incidents": [to_dict(i) for i in state.incidents],
        "action_payloads": [to_dict(a) for a in state.action_payloads],
    }
    json_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

    lines = [f"# Incident Analysis Report — {state.run_id}", ""]
    lines.append("## Summary")
    lines.append(f"- Events parsed: {len(state.events)}")
    lines.append(f"- Signals found: {len(state.signals)}")
    lines.append(f"- Incident candidates found: {len(state.clusters)}")
    lines.append(f"- Incidents found: {len(state.incidents)}")
    lines.append(f"- Ground truth files detected but not used at runtime: {len(state.ground_truth_files)}")
    lines.append("")

    for incident in state.incidents:
        lines.append(f"## {incident.title}")
        lines.append(f"- Severity: {incident.severity}")
        lines.append(f"- Category: {incident.category}")
        lines.append(f"- Confidence: {incident.confidence}")
        lines.append(f"- Affected services: {', '.join(incident.affected_services) or 'unknown'}")
        lines.append(f"- Root cause: {incident.root_cause.get('primary_root_cause')}")
        if incident.rag_sources:
            lines.append(f"- RAG sources: {', '.join(incident.rag_sources)}")
        lines.append("")
        lines.append("### Evidence")
        for ev in incident.root_cause.get("supporting_evidence", [])[:8]:
            lines.append(f"- {ev}")
        lines.append("")
        lines.append("### Immediate Actions")
        for step in incident.remediation.get("immediate_actions", []):
            lines.append(f"- {step}")
        lines.append("")
        lines.append("### Safety Notes")
        for note in incident.remediation.get("safety_notes", []):
            lines.append(f"- {note}")
        lines.append("")

    md_path.write_text("\n".join(lines), encoding="utf-8")

    with open(csv_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["incident_id", "severity", "category", "confidence", "title", "affected_services"])
        writer.writeheader()
        for incident in state.incidents:
            writer.writerow({
                "incident_id": incident.incident_id,
                "severity": incident.severity,
                "category": incident.category,
                "confidence": incident.confidence,
                "title": incident.title,
                "affected_services": ",".join(incident.affected_services),
            })

    return {
        "markdown": str(md_path),
        "json": str(json_path),
        "csv": str(csv_path),
    }


def show_response_summary(response: AnalysisResponse) -> None:
    print(json.dumps(response.summary, indent=2))
    for incident in response.incidents[:10]:
        print(f"\n[{incident.severity}] {incident.category}: {incident.title}")
        print(f"Services: {', '.join(incident.affected_services) or 'unknown'}")
        print(f"Root cause: {incident.root_cause.get('primary_root_cause')}")
        print(f"RAG sources: {', '.join(incident.rag_sources) or 'none'}")


## 12. Run the Notebook

### Option A — Colab upload

```python
from google.colab import files
uploaded = files.upload()
response = analyze_uploaded_logs(list(uploaded.values()))
show_response_summary(response)
```

### Option B — Local/sample ZIP path

```python
response = analyze_uploaded_logs([
    "/mnt/data/devops_raw_log_corpus_v2_all.zip"
], options={"enable_llm": False, "enable_rag": True})
show_response_summary(response)
```


In [ ]:
# Example usage. Uncomment in Colab or local notebook when a sample ZIP is available.
# response = analyze_uploaded_logs(["/mnt/data/devops_raw_log_corpus_v2_all.zip"], options={"enable_llm": False, "enable_rag": True})
# show_response_summary(response)
# response.reports


## 13. v6 Refactor Map to `src/`

| Notebook section | Future module | Owner intent |
|---|---|---|
| Contracts | `src/models/` | Shared Pydantic models |
| Upload boundary | `src/api/` or `src/ui/` | Gradio/FastAPI upload handling |
| Ingestion/parsing | `src/ingest/` | Safe ZIP extraction and normalization |
| Signal extraction | `src/signals/` | Deterministic rule engine |
| Incident candidate builder | `src/clustering/` | Simple grouping first; advanced clustering later |
| RAG retrieval | `src/rag/` | SOP/runbook retrieval |
| Multi-agent reasoning | `src/graph/` + `src/agents/` | LangGraph nodes and fallback logic |
| Action payloads | `src/reporting/` / `src/adapters/` | Slack/JIRA/cookbook previews |
| Exports | `src/reporting/` | Markdown/JSON/CSV outputs |

## What v6 Intentionally Does Not Do Yet

- No real Slack/JIRA writes by default.
- No Weaviate requirement.
- No database persistence.
- No complex trace/dependency clustering.
- No hidden ground truth used during runtime analysis.

These are good future enhancements, but not blockers for the hackathon MVP.
